# Classify Architecture Pain Points — Solution


## Pain Points Table

| # | Service | Category | Pain Point | Measurable Impact |
|---|---------|----------|-----------|-------------------|
| 1 | PostgreSQL (RDS) | Performance | OLTP database used for analytical queries — production workloads compete with reporting | Increased query latency; risk of production outage during heavy analytical scans |
| 2 | Amazon S3 | Performance | Small files (avg 200KB) in JSON format, unpartitioned | Athena/Spark queries scan entire dataset; high per-request overhead; 10x slower than optimized Parquet |
| 3 | Amazon Redshift | Cost | 2-node cluster running 24/7 with no sort keys or distribution keys | ~40% wasted compute; estimated $2,500/mo in unnecessary always-on cost |
| 4 | Amazon Redshift | Performance | Lambda functions write directly to Redshift (small-block inserts), no optimization | Storage fragmentation, slow scans, concurrency contention with Tableau |
| 5 | Tableau → Redshift | Performance | Dashboard refreshes compete with ETL writes for cluster resources | Slow dashboard loads during business hours; users complain of 30s+ wait times |
| 6 | AWS Glue | Cost / Reliability | 4 deprecated jobs still scheduled alongside active ones; no dependency management | Wasted compute ($200+/mo); risk of stale data if deprecated job overwrites active one |
| 7 | (Missing) | Governance | No data catalog or governance layer exists | No lineage, no ownership, no access control beyond IAM; PII exposure risk; teams don't know what data exists |

## Annotated Architecture Diagram

```mermaid
flowchart TD
    subgraph Sources["DATA SOURCES"]
        events["User Events\n(streaming)"]
        billing["Billing\n(daily batch)"]
    end

    subgraph Ingestion["INGESTION"]
        kinesis["Kinesis Streams"]
        lambda["Lambda Functions\n⚠️ Direct writes to Redshift"]
        glue["Glue ETL\n⚠️ 4 deprecated jobs still running"]
    end

    subgraph Storage["STORAGE"]
        s3["S3\n⚠️ Small files, JSON\n⚠️ No partitioning"]
        redshift["Redshift (2-node 24/7)\n⚠️ No sort/dist keys\n⚠️ Small-block inserts\n⚠️ $6,400/mo"]
        postgres["PostgreSQL RDS\n⚠️ Analytics on OLTP"]
    end

    subgraph Consumption["CONSUMPTION"]
        tableau["Tableau\n⚠️ Competes with ETL"]
    end

    subgraph Missing["⚠️ MISSING"]
        catalog["No Data Catalog"]
        governance["No Governance Layer"]
    end

    events --> kinesis --> s3
    kinesis --> lambda --> redshift
    billing --> glue --> s3
    glue --> redshift
    postgres --> tableau
    redshift --> tableau

    style redshift fill:#ffcdd2,stroke:#b71c1c
    style postgres fill:#ffcdd2,stroke:#b71c1c
    style Missing fill:#fff3e0,stroke:#e65100
```